In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [3]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [4]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [5]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'Hi '
                                                                          'John,\n'
                                                                          '\n'
                                                                          'No '
                                                                          'problem—thanks '
                                                                          'for '
                                                                          'the '
                                                                          'heads '
                                                                          'up. '
                                                                          'I’m '
                                                                          'happy '
                                                                          'to '
               

In [6]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'Hi John,\n\nNo problem—thanks for the heads up. I’m happy to reschedule. Are you available at 11:00 AM or 3:00 PM tomorrow? If neither works, please suggest a time that suits you and I’ll adjust.\n\nBest regards,\nSeán'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'Hi John,\\n\\nNo problem—thanks for the heads up. I’m happy to reschedule. Are you available at 11:00 AM or 3:00 PM tomorrow? If neither works, please suggest a time that suits you and I’ll adjust.\\n\\nBest regards,\\nSeán'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject']}]}, id='0d44ed33ccc2189f445df0e6ead7c90b')]


In [7]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

Hi John,

No problem—thanks for the heads up. I’m happy to reschedule. Are you available at 11:00 AM or 3:00 PM tomorrow? If neither works, please suggest a time that suits you and I’ll adjust.

Best regards,
Seán


## Approve

In [6]:
from langgraph.types import Command


In [ ]:

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

## Reject

In [ ]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

In [ ]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

## Edit

In [ ]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

NameError: name 'pprint' is not defined

## 两次 invoke() 是同一个 LLM 会话吗？

两次 `invoke()` 在 **LLM 视角是同一个会话**，因为它们共享相同的 `thread_id` 和 checkpointer state。但在 **LangSmith 里会显示为两条独立 trace**，原因是这两个概念追踪的是不同维度：

| | LangSmith Trace | LangGraph Thread |
|---|---|---|
| **追踪单位** | 一次 `invoke()` 调用 | 一段持久化会话 |
| **标识符** | `run_id`（每次 invoke 自动生成新的） | `thread_id`（用户指定，跨 invoke 共享） |
| **目的** | 记录单次执行的调用链、耗时、token | 持久化 state，支持中断恢复 |

每次调用 `agent.invoke()` 时，内部会生成新的 `run_id` 并向 LangSmith 上报一条新的 trace root，因此两条 trace 在 LangSmith 里是独立的。

**这是正常且合理的**——两次 invoke 中间夹着人工审批这段"现实世界时间"，LangSmith 无法也不应该跨越这段时间合并它们。会话的连续性体现在 LangGraph checkpointer 层（通过 `thread_id` 恢复完整 state），不在 LangSmith 层。

第二次 invoke 结束后，LLM 看到的完整消息历史是连贯的：

```
HumanMessage("Please read my email...")
AIMessage(tool_calls=[read_email])        ← 第一次 LLM 调用
ToolMessage(email content)
AIMessage(tool_calls=[send_email])        ← 第二次 LLM 调用（中断前）
ToolMessage("Email sent")                ← approve 后执行
AIMessage("I've read your email...")     ← 第三次 LLM 调用
```

`Command(resume=...)` 是 LangGraph 的恢复信号，告诉引擎从上次暂停处继续，而非发起新对话。

## Edit invoke 的实际 API 请求解析

以下是 edit 这次 `agent.invoke()` 内部向 LLM 发出的真实 HTTP 请求（通过网络拦截获得）。

```json
{
  "messages": [
    {
      // ① 用户原始任务，来自第一次 invoke() 传入的 HumanMessage
      "content": "Please read my email and send a response immediately. Send the reply now in the same thread.",
      "role": "user"
    },
    {
      // ② 第一轮：LLM 决定先调用 read_email（无需人工审批，直接执行）
      "content": null,
      "role": "assistant",
      "tool_calls": [
        {
          "type": "function",
          "id": "call_9d0tzuPaiyZalD5Oj19lUDfW",
          "function": {
            "name": "read_email",
            "arguments": "{}"
          }
        }
      ]
    },
    {
      // ③ read_email 工具的返回结果（从 state["email"] 读取）
      "content": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John.",
      "role": "tool",
      "tool_call_id": "call_9d0tzuPaiyZalD5Oj19lUDfW"
    },
    {
      // ④ 第二轮：LLM 决定调用 send_email
      //    注意：这里的 arguments 已经是人工 edit 后的版本
      //    原始 LLM 生成的内容（礼貌的重新安排邮件）被替换成了 "This is the last straw, you're fired!"
      //    HumanInTheLoopMiddleware 在执行工具前将 edited_action.args 注入此处
      "content": null,
      "role": "assistant",
      "tool_calls": [
        {
          "type": "function",
          "id": "call_E1qqIvKauLHqc9AVrp3tqyhe",
          "function": {
            "name": "send_email",
            "arguments": "{\"body\": \"This is the last straw, you're fired!\"}"
          }
        }
      ]
    },
    {
      // ⑤ send_email 工具的返回结果（发送成功）
      //    此时 LLM 不知道 body 曾被人工修改过，它看到的就是一次正常的工具调用结果
      "content": "Email sent",
      "role": "tool",
      "tool_call_id": "call_E1qqIvKauLHqc9AVrp3tqyhe"
    }
  ],

  "model": "gpt-5-nano",
  "stream": false,

  // 注册给 LLM 的可用工具列表，LLM 据此决定调用哪个工具及传什么参数
  "tools": [
    {
      "type": "function",
      "function": {
        "name": "read_email",
        "description": "Read an email from the given address.",
        "parameters": { "properties": {}, "type": "object" }
      }
    },
    {
      "type": "function",
      "function": {
        "name": "send_email",
        "description": "Send an email to the given address with the given subject and body.",
        "parameters": {
          "properties": { "body": { "type": "string" } },
          "required": ["body"],
          "type": "object"
        }
      }
    }
  ]
}
```

### 关键观察

1. **完整历史透传**：LLM 收到了从头开始的完整消息链（① → ⑤），这证明两次 `invoke()` 通过 checkpointer + `thread_id` 共享了同一份 state，LLM 视角是连续会话。

2. **edit 的注入点在 ④**：人工修改的内容（`"This is the last straw, you're fired!"`）被 `HumanInTheLoopMiddleware` 替换进了 `AIMessage` 的 `tool_calls.arguments`，然后才真正执行工具。LLM 本轮调用（第三次）看到的 ④ 已经是修改后的版本。

3. **LLM 对修改无感知**：从 ④⑤ 来看，LLM 看到的是一次"正常"的工具调用和返回，完全不知道 body 曾被人工篡改。这是 HITL 中间件设计的核心——**人介入的痕迹对 LLM 透明**。

4. **`system_prompt` 未出现**：此请求中没有 `system` 消息，说明本例 `create_agent` 未配置 `system_prompt`。